# Demo: Synonym Table Schema

Demonstrates creating a synonym table and rows using the fixed schema defined in `scripts/APIs_pipe/schema.py`.

Run from the project root.

In [1]:
import pandas as pd

from scripts.APIs_pipe.schema import (
    SYNONYM_COLUMNS,
    UNAVAILABLE,
    empty_synonym_table,
    make_synonym_row,
)

## 1. Inspect the schema

In [2]:
print("Schema columns:")
for col in SYNONYM_COLUMNS:
    print(f"  {col}")

Schema columns:
  Source Name
  Kingdom
  Phylum
  Class
  Family
  Subfamily
  Genus
  Species
  Source Species ID
  Author
  Publication Name
  Publication Year
  Source Link
  GBIF Accepted Status


## 2. Create an empty table

In [3]:
table = empty_synonym_table()
print(f"Empty table shape: {table.shape}")
table

Empty table shape: (0, 14)


,Source Name,Kingdom,Phylum,Class,Family,Subfamily,Genus,Species,Source Species ID,Author,Publication Name,Publication Year,Source Link,GBIF Accepted Status


## 3. Create a fully-specified row

In [4]:
row_full = make_synonym_row(
    **{
        "Source Name": "GBIF",
        "Kingdom": "Fungi",
        "Phylum": "Basidiomycota",
        "Class": "Agaricomycetes",
        "Family": "Amanitaceae",
        "Subfamily": UNAVAILABLE,
        "Genus": "Amanita",
        "Species": "muscaria",
        "Source Species ID": "8168319",
        "Author": "(L.) Lam.",
        "Publication Name": "Encycl. Méth. Bot. 1(1): 111",
        "Publication Year": "1783",
        "Source Link": "https://www.gbif.org/species/8168319",
        "GBIF Accepted Status": "Accepted",
    }
)
row_full

{'Source Name': 'GBIF',
 'Kingdom': 'Fungi',
 'Phylum': 'Basidiomycota',
 'Class': 'Agaricomycetes',
 'Family': 'Amanitaceae',
 'Subfamily': 'U',
 'Genus': 'Amanita',
 'Species': 'muscaria',
 'Source Species ID': '8168319',
 'Author': '(L.) Lam.',
 'Publication Name': 'Encycl. Méth. Bot. 1(1): 111',
 'Publication Year': '1783',
 'Source Link': 'https://www.gbif.org/species/8168319',
 'GBIF Accepted Status': 'Accepted'}

## 4. Create a partial row (missing fields default to `"U"`)

In [5]:
row_partial = make_synonym_row(
    **{
        "Source Name": "mushroomobs",
        "Genus": "Amanita",
        "Species": "muscaria",
        "Source Species ID": "16015",
        "Source Link": "https://mushroomobserver.org/name/show_name/16015",
        "GBIF Accepted Status": UNAVAILABLE,
    }
)
row_partial

{'Source Name': 'mushroomobs',
 'Kingdom': 'U',
 'Phylum': 'U',
 'Class': 'U',
 'Family': 'U',
 'Subfamily': 'U',
 'Genus': 'Amanita',
 'Species': 'muscaria',
 'Source Species ID': '16015',
 'Author': 'U',
 'Publication Name': 'U',
 'Publication Year': 'U',
 'Source Link': 'https://mushroomobserver.org/name/show_name/16015',
 'GBIF Accepted Status': 'U'}

## 5. Append both rows to the table

In [6]:
table = pd.concat(
    [table, pd.DataFrame([row_full, row_partial])],
    ignore_index=True,
)
print(f"Table shape: {table.shape}")
table

Table shape: (2, 14)


,Source Name,Kingdom,Phylum,Class,Family,Subfamily,Genus,Species,Source Species ID,Author,Publication Name,Publication Year,Source Link,GBIF Accepted Status
0,GBIF,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,U,Amanita,muscaria,8168319,(L.) Lam.,Encycl. Méth. Bot. 1(1): 111,1783,https://www.gbif.org/species/8168319,Accepted
1,mushroomobs,U,U,U,U,U,Amanita,muscaria,16015,U,U,U,https://mushroomobserver.org/name/show_name/16015,U


## 6. Demonstrate validation errors

In [ ]:
test_cases = [
    ("bad Publication Year", {"Source Name": "GBIF", "Genus": "Amanita", "Species": "muscaria", "GBIF Accepted Status": "Accepted", "Source Link": "https://example.com", "Publication Year": "83"}),
    ("bad Source Link",      {"Source Name": "GBIF", "Genus": "Amanita", "Species": "muscaria", "GBIF Accepted Status": "Accepted", "Source Link": "not-a-url"}),
    ("bad GBIF Status",      {"Source Name": "GBIF", "Genus": "Amanita", "Species": "muscaria", "GBIF Accepted Status": "Unknown", "Source Link": "https://example.com"}),
    ("multi-word Genus",     {"Source Name": "GBIF", "Genus": "Amanita muscaria", "Species": "muscaria", "GBIF Accepted Status": "Accepted", "Source Link": "https://example.com"}),
]

for label, kwargs in test_cases:
    try:
        make_synonym_row(**kwargs)
        print(f"  {label}: no error (unexpected)")
    except ValueError as e:
        print(f"  {label}: ValueError — {e}")

  bad Publication Year: ValueError raised correctly — 'Publication Year' must be a 4-digit year string or 'U', got '83'
  bad Source Link: ValueError raised correctly — 'Source Link' must start with 'http://' or 'https://', or be 'U', got 'not-a-url'
  bad GBIF Status: ValueError raised correctly — 'GBIF Accepted Status' must be one of {'Synonym', 'Accepted', 'U'}, got 'Unknown'
  multi-word Genus: ValueError raised correctly — 'Genus' must be a single word (no whitespace) or 'U', got 'Amanita muscaria'
